# ANÁLISIS EXPLORATORIO DE DATOS (EDA)

En esta sección ejecutaremos el análisis estadístico descriptivo e inferencial sobre el dataset final procesado. Utilizaremos exclusivamente **Pandas** y **NumPy** para extraer los estadísticos clave y evaluar las correlaciones entre el rendimiento en pista, la estrategia y el resultado final de carrera.


## 1. Carga del Dataset Procesado y Cálculo de Estadísticos Descriptivos

Analizamos las medidas de tendencia central (media y mediana) y dispersión (desviación estándar, mínimos y máximos) para las variables cuantitativas principales.

In [1]:
import pandas as pd
import numpy as np

# Cargar el conjunto de datos
df = pd.read_csv('f1_data.csv')
df.head()

/var/folders/9m/tm3_26dd11sfl1btdhw5wkwh0000gn/T/ipykernel_54680/3573758656.py:5: DtypeWarning: Columns (11,15,19,28,44) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('f1_data.csv')


,raceId,driverId,lap,position_lap,time_lap,milliseconds_lap,resultId,constructorId,number,grid,...,alt,stop,pit_duration,pit_milliseconds,driver_full_name,driver_age,lap_seconds,positions_gained,is_pit_stop,pit_seconds
0,479,137,1,1,1:42.085,102085,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,102.085,-10.0,False,NaN
1,479,137,2,2,1:36.287,96287,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,96.287,-10.0,False,NaN
2,479,137,3,2,1:34.627,94627,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,94.627,-10.0,False,NaN
3,479,137,4,2,1:34.041,94041,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,94.041,-10.0,False,NaN
4,479,137,5,2,1:33.699,93699,11437,34,1.0,1.0,...,678,NaN,NaN,NaN,Nelson Piquet,30,93.699,-10.0,False,NaN


In [2]:
# 1. Selección de columnas cuantitativas principales 
cols_eda = ['lap_seconds', 'driver_age', 'grid', 'positionOrder', 'positions_gained', 'pit_seconds']

# 2. Cáaculo de estadísticos descriptivos
estadisticos = df[cols_eda].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']].round(2)
estadisticos.columns = ['Recuento', 'Media', 'Desv. Est.', 'Mínimo', 'Mediana (Q2)', 'Máximo']

print(estadisticos)

                  Recuento  Media  Desv. Est.  Mínimo  Mediana (Q2)   Máximo
lap_seconds       278678.0  95.64       78.12   65.26         90.71  7507.55
driver_age        278678.0  29.89        4.44   20.00         30.00    43.00
grid              278678.0  12.63        7.42    1.00         12.00    29.00
positionOrder     278678.0   9.99        6.04    1.00          9.00    27.00
positions_gained  278678.0   2.64        7.10  -25.00          3.00    22.00
pit_seconds         2424.0  30.14       30.42   12.90         25.50  1004.72


In [3]:
# Análisis del factor "Pole Position a Victoria" 
races_unicas = df[['raceId', 'driverId', 'grid', 'positionOrder']].drop_duplicates()

total_poles = np.sum(races_unicas['grid'] == 1)
total_pole_wins = np.sum((races_unicas['grid'] == 1) & (races_unicas['positionOrder'] == 1))
pct_conversion = (total_pole_wins / total_poles) * 100

print(f"Carreras con Pole analizadas: {total_poles}")
print(f"Victorias desde la Pole Position: {total_pole_wins}")
print(f"Porcentaje de conversión a victoria: {pct_conversion}%")

Carreras con Pole analizadas: 244
Victorias desde la Pole Position: 97
Porcentaje de conversión a victoria: 39.75409836065574%


## 2.Análisis de interés

A continuación dmaos un paso más dentro del estudio del conjunto de datos que disponemos. Ampliamos el análisis exploratorio ejecutando cuatro estudios estadísticos independientes sobre el dataset unificado (`f1_data.csv`), los cuales responden a preguntas muy interesantes que pueden describir y relacionar el rendimiento con los diferentes factores, no solo operacionales, que componen el dataset.

1. **Estudio de Remontadas:** Agregación de posiciones ganadas netas por piloto.
2. **Análisis de Fiabilidad (DNFs):** Tasa de abandonos por escudería usando filtros.
3. **Análisis de Dispersion de Tiempos en Pista:** Medición de consistencia de ritmo por temporada.
4. **Percentiles Operativos de Boxes:** Cálculo de $P_{25}$, $P_{50}$ y $P_{75}$ en paradas en boxes.

### ESTUDIO 1: ANÁLISIS DE REMONTADAS Y RECUPERACIÓN DE POSICIONES 

In [4]:
remontadas = df[['raceId', 'driver_full_name', 'positions_gained', 'grid']].drop_duplicates()
analisis_remontadas = remontadas.groupby('driver_full_name').agg(
    carreras=('positions_gained', 'count'),
    media_pos_ganadas=('positions_gained', 'mean'),
    max_remontada=('positions_gained', 'max'),
    std_remontada=('positions_gained', 'std')
).query('carreras >= 30').sort_values('media_pos_ganadas', ascending=False).head(5).round(2)

print(analisis_remontadas)


                      carreras  media_pos_ganadas  max_remontada  \
driver_full_name                                                   
Jonathan Palmer             81               6.59           19.0   
Christian Fittipaldi        40               6.35           16.0   
Christian Danner            34               6.21           22.0   
Marc Surer                  55               5.22           20.0   
Gabriele Tarquini           35               5.17           18.0   

                      std_remontada  
driver_full_name                     
Jonathan Palmer                6.68  
Christian Fittipaldi           6.32  
Christian Danner               6.51  
Marc Surer                     7.66  
Gabriele Tarquini              6.77  


### ESTUDIO 2: FIABILIDAD Y TASA DE ABANDONOS (DNFs) POR ESCUDERÍA 
Nota para est eestudio, statusId == 1 representa coche clasificado/finalizado


In [6]:
fiabilidad = df[['raceId', 'driverId', 'constructor_name', 'statusId']].drop_duplicates()
resumen_fiabilidad = fiabilidad.groupby('constructor_name').agg(
    total_participaciones=('statusId', 'count'),
    abandonos=('statusId', lambda x: np.sum(x != 1))
)
resumen_fiabilidad['pct_abandonos'] = (resumen_fiabilidad['abandonos'] / resumen_fiabilidad['total_participaciones'] * 100).round(2)
top_fiabilidad = resumen_fiabilidad.query('total_participaciones >= 100').sort_values('pct_abandonos').head(5)
print(top_fiabilidad)


                  total_participaciones  abandonos  pct_abandonos
constructor_name                                                 
McLaren                             473        219          46.30
Williams                            480        268          55.83
Ferrari                             470        278          59.15
Benetton                            312        200          64.10
Renault                             156        113          72.44


### ESTUDIO 3: EVOLUCIÓN HISTÓRICA DEL RITMO DE CARRERA Y CONSISTENCIA 

In [ ]:
consistencia_temporada = df.groupby('year').agg(
    vueltas_totales=('lap_seconds', 'count'),
    tiempo_medio_vuelta=('lap_seconds', 'mean'),
    desviacion_ritmo=('lap_seconds', 'std')
).tail(10).round(2)

print("\n=== ESTUDIO 3: RITMO Y DESVIACIÓN TÍPICA ÚLTIMAS 10 TEMPORADAS ===")
print(consistencia_temporada)


### ESTUDIO 4: PERCENTILES Y DISTRIBUCIÓN EN PIT STOPS CON NUMPY 

In [7]:
pit_vueltas = df[df['is_pit_stop'] == True]['pit_seconds'].dropna()

p25 = np.percentile(pit_vueltas, 25).round(2)
p50 = np.percentile(pit_vueltas, 50).round(2) # Mediana
p75 = np.percentile(pit_vueltas, 75).round(2)

print(f"Percentil 25 (Parada rápida): {p25} s")
print(f"Percentil 50 (Mediana): {p50} s")
print(f"Percentil 75 (Parada con tráfico/incidencia): {p75} s")

Percentil 25 (Parada rápida): 22.23 s
Percentil 50 (Mediana): 25.5 s
Percentil 75 (Parada con tráfico/incidencia): 31.43 s
